# Dotrenowanie Stable Diffusion (LoRA) na rasach psów — Google Colab

Notebook przygotowany do uruchomienia w **Google Colab z GPU** (Runtime → Change
runtime type → T4 GPU). Dotrenowuje model **Stable Diffusion v1.5** metodą **LoRA**
na 5 rasach psów ze zbioru **Stanford Dogs**, a następnie porównuje generację
**przed** i **po** treningu.

> Uwaga: treningu nie da się sensownie wykonać na CPU — wymaga karty GPU.
> Dlatego notebook jest przeznaczony na Colab. Poniższe komórki należy uruchomić
> po kolei.

## 1. Instalacja zależności i repozytorium diffusers

In [ ]:
!pip install -q diffusers["torch"] transformers accelerate peft bitsandbytes ftfy
!git clone https://github.com/huggingface/diffusers
%cd diffusers/examples/text_to_image
!pip install -q -r requirements.txt

## 2. Pobranie zbioru Stanford Dogs

Wymaga pliku `kaggle.json` (Account → Create New API Token na kaggle.com).

In [ ]:
from google.colab import files
files.upload()  # wgraj kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d jessicali9530/stanford-dogs-dataset
!unzip -q stanford-dogs-dataset.zip -d dane
# Struktura: dane/images/Images/<rasy>/  -> ujednolicamy do dane/Images
!mv dane/images/Images dane/Images 2>/dev/null || true

## 3. Przygotowanie 5 ras i pliku metadata.jsonl

Wybrane rasy: **beagle, golden retriever, german shepherd, siberian husky, boxer**.
Każdy obraz dostaje prompt o spójnej strukturze `photo of <rasa> dog`.

In [ ]:
import json, shutil
from pathlib import Path

ZRODLO = Path('dane/Images')
CEL = Path('dane/zbior_treningowy')
CEL.mkdir(parents=True, exist_ok=True)
RASY = {'beagle':'beagle','golden_retriever':'golden retriever',
        'German_shepherd':'german shepherd','Siberian_husky':'siberian husky','boxer':'boxer'}
MAKS = 80

meta = []
for frag, nazwa in RASY.items():
    folder = next((f for f in ZRODLO.iterdir() if frag.lower() in f.name.lower()), None)
    if not folder: print('brak:', nazwa); continue
    for img in [p for p in folder.iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')][:MAKS]:
        nn = f"{nazwa.replace(' ','_')}_{img.name}"
        shutil.copy2(img, CEL/nn)
        meta.append({'file_name': nn, 'text': f'photo of {nazwa} dog'})
with open(CEL/'metadata.jsonl','w') as f:
    for m in meta: f.write(json.dumps(m)+'\n')
print('Par obraz-prompt:', len(meta))

## 4. Generacja PRZED treningiem (model bazowy)

Zapisujemy obrazy bazowego SD v1.5, żeby mieć punkt odniesienia.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

MODEL = 'runwayml/stable-diffusion-v1-5'
pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, safety_checker=None).to('cuda')

import os; os.makedirs('wyniki/przed', exist_ok=True)
for rasa in ['beagle','golden retriever','german shepherd','siberian husky','boxer']:
    img = pipe(f'a photo of {rasa} dog, highly detailed, realistic', num_inference_steps=30, guidance_scale=7.5).images[0]
    img.save(f"wyniki/przed/{rasa.replace(' ','_')}.png")
print('Zapisano obrazy bazowe w wyniki/przed/')

## 5. Trening LoRA (jedna LoRA na wszystkich 5 rasach)

Używamy gotowego skryptu `train_text_to_image_lora.py` z diffusers, uruchamianego
przez `accelerate`. Hiperparametry dobrane pod ograniczenia pamięci GPU.

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path='runwayml/stable-diffusion-v1-5' \
  --train_data_dir='/content/diffusers/examples/text_to_image/dane/zbior_treningowy' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 --gradient_checkpointing \
  --max_train_steps=1000 --learning_rate=1e-04 \
  --lr_scheduler='constant' --lr_warmup_steps=0 --seed=42 \
  --mixed_precision='fp16' --dataloader_num_workers=2 \
  --output_dir='models/lora_5_ras'

## 6. Generacja PO treningu (z wagami LoRA) i porównanie

In [ ]:
pipe.load_lora_weights('models/lora_5_ras')
os.makedirs('wyniki/po', exist_ok=True)
for rasa in ['beagle','golden retriever','german shepherd','siberian husky','boxer']:
    img = pipe(f'a photo of {rasa} dog, highly detailed, realistic', num_inference_steps=30, guidance_scale=7.5).images[0]
    img.save(f"wyniki/po/{rasa.replace(' ','_')}.png")
print('Zapisano obrazy po treningu w wyniki/po/ — porównaj z wyniki/przed/')

In [ ]:
# Zestawienie przed/po obok siebie
import matplotlib.pyplot as plt
from PIL import Image
rasy = ['beagle','golden_retriever','german_shepherd','siberian_husky','boxer']
fig, ax = plt.subplots(2, len(rasy), figsize=(20,8))
for i, r in enumerate(rasy):
    ax[0,i].imshow(Image.open(f'wyniki/przed/{r}.png')); ax[0,i].set_title(f'PRZED: {r}'); ax[0,i].axis('off')
    ax[1,i].imshow(Image.open(f'wyniki/po/{r}.png')); ax[1,i].set_title(f'PO: {r}'); ax[1,i].axis('off')
plt.tight_layout(); plt.show()